# Chamber Septa Builder

Build curved internal chamber walls inside the shell.

Conceptually, these are septa: internal partitions laid down at intervals as the shell grows. In a Nautilus-like shell, the animal moves forward into the newest body chamber, leaving older chambers behind, separated by curved walls.

This code takes the existing shell surface mesh and inserts chamber walls at selected aperture rings.

Each septum is built as a shallow concave bowl:

- the outer edge matches an existing aperture ring
- nested smaller rings fill the aperture
- the centre is pushed slightly backward along the growth direction
- the result is a curved chamber wall rather than a flat disc

Chamber colouring is varied by growth age:

- older inner chambers are darker
- newer outer chambers are lighter


In [ ]:
import numpy as np

In [ ]:
def unpack_outer_shell_mesh(shell_mesh):
    """
    Extract vertex coordinate arrays from a shell mesh.

    :param shell_mesh: Tuple returned by build_shell_mesh
    :return: X, Y, Z coordinate arrays
    """
    X, Y, Z, _, _, _, _ = shell_mesh
    return X, Y, Z

In [ ]:
def get_aperture_ring(X, Y, Z, ring_index, aperture_points):
    """
    Extract one aperture ring from the shell mesh.

    :param X: Shell mesh x coordinates
    :param Y: Shell mesh y coordinates
    :param Z: Shell mesh z coordinates
    :param ring_index: Aperture ring index along the growth path
    :param aperture_points: Number of vertices per aperture ring
    :return: ring_x, ring_y, ring_z arrays
    """
    start = ring_index * aperture_points
    end = start + aperture_points

    return X[start:end], Y[start:end], Z[start:end]

In [ ]:
def ring_centre(ring_x, ring_y, ring_z):
    """
    Estimate the centre point of an aperture ring.

    :param ring_x: Ring x coordinates
    :param ring_y: Ring y coordinates
    :param ring_z: Ring z coordinates
    :return: centre_x, centre_y, centre_z
    """
    return np.mean(ring_x), np.mean(ring_y), np.mean(ring_z)

In [ ]:
def estimate_growth_direction(X, Y, Z, ring_index, aperture_points):
    """
    Estimate the local forward growth direction using neighbouring rings.

    :param X: Shell mesh x coordinates
    :param Y: Shell mesh y coordinates
    :param Z: Shell mesh z coordinates
    :param ring_index: Current aperture ring index
    :param aperture_points: Number of vertices per aperture ring
    :return: Normalised direction vector
    """
    prev_x, prev_y, prev_z = get_aperture_ring(
        X, Y, Z, ring_index - 1, aperture_points
    )
    next_x, next_y, next_z = get_aperture_ring(
        X, Y, Z, ring_index + 1, aperture_points
    )

    direction = np.array([
        np.mean(next_x) - np.mean(prev_x),
        np.mean(next_y) - np.mean(prev_y),
        np.mean(next_z) - np.mean(prev_z),
    ])

    norm = np.linalg.norm(direction)

    if norm == 0:
        return np.array([0, 0, 1])

    return direction / norm

In [ ]:
def estimate_aperture_radius(ring_x, ring_y, ring_z, centre_x, centre_y, centre_z):
    """
    Estimate the average radius of an aperture ring.

    :param ring_x: Ring x coordinates
    :param ring_y: Ring y coordinates
    :param ring_z: Ring z coordinates
    :param centre_x: Ring centre x coordinate
    :param centre_y: Ring centre y coordinate
    :param centre_z: Ring centre z coordinate
    :return: Mean distance from centre to ring vertices
    """
    return np.mean(
        np.sqrt(
            (ring_x - centre_x)**2 +
            (ring_y - centre_y)**2 +
            (ring_z - centre_z)**2
        )
    )

In [ ]:
def chamber_colour(ring_index, n_spiral):
    """
    Compute chamber colour based on growth age.

    :param ring_index: Aperture ring index where the chamber is inserted
    :param n_spiral: Total number of shell aperture rings
    :return: Scalar colour/intensity value
    """
    growth_fraction = ring_index / (n_spiral - 1)
    return 0.4 + 0.6 * growth_fraction

In [ ]:
def build_single_septum(
    ring_x,
    ring_y,
    ring_z,
    direction,
    septum_rings,
    septum_depth,
    aperture_points,
    colour_value
):
    """
    Build one curved septum as nested aperture rings.

    :param ring_x: Outer aperture ring x coordinates
    :param ring_y: Outer aperture ring y coordinates
    :param ring_z: Outer aperture ring z coordinates
    :param direction: Normalised local growth direction
    :param septum_rings: Number of nested rings used to fill the septum
    :param septum_depth: Strength of septum concavity
    :param aperture_points: Number of points around each aperture ring
    :param colour_value: Colour value assigned to this septum
    :return: Xc, Yc, Zc, Cc arrays for this septum
    """
    centre_x, centre_y, centre_z = ring_centre(ring_x, ring_y, ring_z)

    aperture_radius = estimate_aperture_radius(
        ring_x, ring_y, ring_z,
        centre_x, centre_y, centre_z
    )

    Xc, Yc, Zc, Cc = [], [], [], []

    for ring_idx in range(septum_rings + 1):
        t = ring_idx / septum_rings
        curve_offset = -septum_depth * aperture_radius * (1 - t**2)

        for p in range(aperture_points):
            px = centre_x + t * (ring_x[p] - centre_x)
            py = centre_y + t * (ring_y[p] - centre_y)
            pz = centre_z + t * (ring_z[p] - centre_z)

            px += curve_offset * direction[0]
            py += curve_offset * direction[1]
            pz += curve_offset * direction[2]

            Xc.append(px)
            Yc.append(py)
            Zc.append(pz)
            Cc.append(colour_value)

    return Xc, Yc, Zc, Cc

In [ ]:
def stitch_septum_surface(Ic, Jc, Kc, base_index, septum_rings, aperture_points):
    """
    Stitch nested septum rings into triangular faces.

    :param Ic: Triangle index list I
    :param Jc: Triangle index list J
    :param Kc: Triangle index list K
    :param base_index: Starting vertex index for this septum
    :param septum_rings: Number of nested septum rings
    :param aperture_points: Number of vertices per ring
    """
    for ring_idx in range(septum_rings):
        ring_a = base_index + ring_idx * aperture_points
        ring_b = base_index + (ring_idx + 1) * aperture_points

        for p in range(aperture_points):
            a0 = ring_a + p
            a1 = ring_a + ((p + 1) % aperture_points)
            b0 = ring_b + p
            b1 = ring_b + ((p + 1) % aperture_points)

            Ic.append(a0)
            Jc.append(b0)
            Kc.append(a1)

            Ic.append(a1)
            Jc.append(b0)
            Kc.append(b1)

In [ ]:
def build_chamber_septa(
    theta,
    shell_mesh,
    aperture_points=32,
    chamber_spacing=0.65,
    septum_rings=8,
    septum_depth=0.35
):
    """
    Build curved internal chamber walls inside the shell.

    :param theta: Spiral angle values representing growth progression
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param aperture_points: Number of vertices around each aperture ring
    :param chamber_spacing: Angular distance between successive chamber walls
    :param septum_rings: Number of nested rings used to fill each septum
    :param septum_depth: Strength of concavity in each chamber wall
    :return: Chamber mesh arrays Xc, Yc, Zc, Cc and triangle index arrays Ic, Jc, Kc
    """
    X, Y, Z = unpack_outer_shell_mesh(shell_mesh)

    n_spiral = len(theta)

    Xc, Yc, Zc, Cc = [], [], [], []
    Ic, Jc, Kc = [], [], []

    last_chamber_theta = theta[0]

    for s in range(1, n_spiral - 1):
        if theta[s] - last_chamber_theta < chamber_spacing:
            continue

        ring_x, ring_y, ring_z = get_aperture_ring(
            X, Y, Z,
            ring_index=s,
            aperture_points=aperture_points
        )

        direction = estimate_growth_direction(
            X, Y, Z,
            ring_index=s,
            aperture_points=aperture_points
        )

        colour_value = chamber_colour(s, n_spiral)

        base_index = len(Xc)

        sx, sy, sz, sc = build_single_septum(
            ring_x,
            ring_y,
            ring_z,
            direction,
            septum_rings=septum_rings,
            septum_depth=septum_depth,
            aperture_points=aperture_points,
            colour_value=colour_value
        )

        Xc.extend(sx)
        Yc.extend(sy)
        Zc.extend(sz)
        Cc.extend(sc)

        stitch_septum_surface(
            Ic, Jc, Kc,
            base_index=base_index,
            septum_rings=septum_rings,
            aperture_points=aperture_points
        )

        last_chamber_theta = theta[s]

    return np.array(Xc), np.array(Yc), np.array(Zc), np.array(Cc), Ic, Jc, Kc